# SensorLoc — GPU training on Kaggle

This notebook trains the **IMU → 24 body-region** classifier (`train.py`, LOSO) using a Kaggle **GPU**. Preprocessing AMASS stays on your **local** machine; you upload the resulting `.npz` (or use the **smoke test** below with no dataset upload).

---

## Before you run

1. **Accelerator:** Notebook settings → **GPU** (e.g. T4).
2. **Add input data:** create a Kaggle Dataset that **always** includes `train.py`, `model.py`, `smpl_regions.py`. For **full** training, also add `dataset.npz` (from local `preprocess_amass.py`). For **smoke test only**, the three `.py` files are enough — `dataset.npz` is generated inside the notebook.
3. **Attach that dataset** to this notebook (right panel → **Add Data**).
4. Set `INPUT_DIR` in the config cell to match your dataset slug (path under `/kaggle/input/<slug>/`).

---

## Local → Kaggle workflow (full pipeline)

| Step | Where | Action |
|------|--------|--------|
| 1 | **Local** | `pip install torch scipy numpy` (CPU ok) |
| 2 | **Local** | Run `preprocess_amass.py` → `data/dataset.npz` |
| 3 | **Local** | Zip `dataset.npz` + `train.py` + `model.py` + `smpl_regions.py` → upload as Kaggle Dataset |
| 4 | **Kaggle** | Run this notebook (full training section) |
| 5 | **Kaggle** | Download `sensorloc_outputs.zip` from **Output** |
| 6 | **Local** | `evaluate.py` with the same `dataset.npz` + downloaded `best_model_fold*.pt` |

**Smoke test:** skips AMASS and large uploads; generates a tiny synthetic `.npz` in `/kaggle/working` and runs a short train to verify GPU and code paths.

## 1. Config

- `SMOKE_TEST` — smoke vs real `dataset.npz`.
- `USE_MULTI_GPU` — set `True` only with **2+ GPUs**; ETA + training both use `DataParallel` (must match).

In [ ]:
import os
import sys
import shutil

# ---------------------------------------------------------------------------
# Toggle smoke vs full training
# ---------------------------------------------------------------------------
SMOKE_TEST = True

# 2+ GPUs (e.g. 2×T4): True → ETA benchmark + train.py use nn.DataParallel. Kaggle usually 1 GPU.
USE_MULTI_GPU = False

# Kaggle dataset folder (slug after /kaggle/input/) — only used when SMOKE_TEST is False
# Example: if dataset is named "sensorloc-train", path is "/kaggle/input/sensorloc-train"
INPUT_DIR = "/kaggle/input/sensorloc-train"

WORK_ROOT = "/kaggle/working"
CODE_DIR = os.path.join(WORK_ROOT, "sensorloc_code")
CKPT_DIR = os.path.join(WORK_ROOT, "checkpoints")
os.makedirs(CODE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print("SMOKE_TEST:", SMOKE_TEST)
print("USE_MULTI_GPU:", USE_MULTI_GPU)
print("CODE_DIR:", CODE_DIR)
print("CKPT_DIR:", CKPT_DIR)

## 2. PyTorch + CUDA check (read if you use older GPUs)

Kaggle’s default **PyTorch 2.10+cu12** builds often **omit kernels for Pascal (sm_60)** — e.g. **Tesla P100** — which causes `no kernel image is available` even when `cuda: True`.

The next cell **detects a broken CUDA tensor op**, then **reinstalls PyTorch cu118** (typically includes sm_60). After it runs you **must restart the session** and re-run from the top.

In [ ]:
# Uncomment if needed:
# !pip install -q numpy

import subprocess
import sys

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


def _cuda_tensor_works() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        torch.randn(2, 3, device="cuda")
        return True
    except Exception:
        return False


if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", cap)

if not torch.cuda.is_available():
    print("No GPU — training will use CPU (slow).")
elif _cuda_tensor_works():
    print("CUDA tensor test: OK")
else:
    print("CUDA is visible but kernels fail (common on P100 + new PyTorch).")
    print("Installing PyTorch 2.2.2 + cu118 (includes sm_60)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"],
        check=False,
    )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch==2.2.2",
            "torchvision==0.17.2",
            "torchaudio==2.2.2",
            "--index-url",
            "https://download.pytorch.org/whl/cu118",
        ]
    )
    print("\n>>> Restart the notebook session (Session → Restart Session), then run all cells again. <<<")

## 3. Copy project code into `/kaggle/working`

**Smoke test:** copies from `INPUT_DIR` if those three `.py` files exist there; otherwise expects you to have attached the same dataset or to paste files manually.

**Tip:** Put `train.py`, `model.py`, `smpl_regions.py` inside your Kaggle Dataset next to `dataset.npz` so one attachment supplies everything.

In [ ]:
REQUIRED = ["train.py", "model.py", "smpl_regions.py"]

def copy_code_from(src_dir: str, dst_dir: str) -> None:
    for name in REQUIRED:
        s, d = os.path.join(src_dir, name), os.path.join(dst_dir, name)
        if not os.path.isfile(s):
            raise FileNotFoundError(
                f"Missing {name} in {src_dir}. Add train.py, model.py, smpl_regions.py to your Kaggle input dataset."
            )
        shutil.copy2(s, d)
        print("copied", name)


if not os.path.isdir(INPUT_DIR):
    raise FileNotFoundError(
        f"INPUT_DIR does not exist: {INPUT_DIR}\n"
        "Attach your Kaggle Dataset and set INPUT_DIR to /kaggle/input/<your-dataset-name>."
    )

copy_code_from(INPUT_DIR, CODE_DIR)
sys.path.insert(0, CODE_DIR)
print("Ready — code in", CODE_DIR)

## 4a. Smoke test — generate synthetic `dataset.npz` (no AMASS)

This **mirrors** local smoke data: `python preprocess_amass.py --smoke_test` → `data/smoke_dataset.npz` (random Gaussian windows, 480 samples, 24 classes, 4 subjects).

**Alternative:** run that command locally and upload `smoke_dataset.npz` to your Kaggle Dataset; then set `SMOKE_TEST = False` and name the file `dataset.npz`, or change the notebook to point at `smoke_dataset.npz`.

When `SMOKE_TEST = False`, this cell only sets `DATA_PATH` to your uploaded `dataset.npz`.

In [ ]:
import numpy as np

NUM_REGIONS = 24

if SMOKE_TEST:
    rng = np.random.default_rng(42)
    N_per = 20
    window = 120
    total = N_per * NUM_REGIONS
    X = rng.standard_normal((total, 6, window)).astype(np.float32)
    y = np.repeat(np.arange(NUM_REGIONS, dtype=np.int32), N_per)
    subject_ids = (np.arange(total) // (N_per * NUM_REGIONS // 4)).astype(np.int32)
    smoke_path = os.path.join(WORK_ROOT, "smoke_dataset.npz")
    np.savez_compressed(smoke_path, X=X, y=y, subject_ids=subject_ids)
    print("Smoke dataset written:", smoke_path, X.shape)
    DATA_PATH = smoke_path
else:
    DATA_PATH = os.path.join(INPUT_DIR, "dataset.npz")
    if not os.path.isfile(DATA_PATH):
        raise FileNotFoundError(
            f"Expected {DATA_PATH}. Add dataset.npz from local preprocess to your Kaggle dataset."
        )
    print("Full dataset:", DATA_PATH)

print("DATA_PATH =", DATA_PATH)

## 4b. Smoke test — training command notes

- Do **not** combine `--smoke_test` with `--data .../smoke_dataset.npz` in this repo (it can rewrite the path incorrectly).
- Here we call `train.py` with an explicit `--data` path and short `--epochs` only.
- Training uses **GPU** when `torch.cuda.is_available()` (`train.py --device cuda`).

## 5. Pre-training ETA check (recommended)

This cell estimates runtime **before** full LOSO training:

- Reads `subject_ids` from `DATA_PATH` to infer number of folds.
- Calculates training batches per fold from your current `batch_size`.
- Benchmarks a few train steps on one fold to estimate `sec/step` (uses **`nn.DataParallel`** when **`USE_MULTI_GPU`** and 2+ GPUs — same as `train.py --multi_gpu`).
- Prints rough total runtime for your chosen `epochs`.

Use this to tune `epochs` / `batch_size` before launching a long run.

In [ ]:
import time
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from train import normalise, wrap_data_parallel
from model import build_model

# Keep defaults aligned with the training cell below
if SMOKE_TEST:
    epochs = 5
    arch = "resnet"
else:
    epochs = 50
    arch = "resnet"

batch_size = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) Dataset-based static estimate (batches per fold)
d = np.load(DATA_PATH)
X = d["X"].astype(np.float32)
y = d["y"].astype(np.int64)
sids = d["subject_ids"].astype(np.int64)
subjects = np.unique(sids)

steps_per_epoch_all_folds = 0
for subj in subjects:
    n_train = int((sids != subj).sum())
    steps = (n_train + batch_size - 1) // batch_size
    steps_per_epoch_all_folds += steps

# 2) Micro-benchmark: a few train steps on one fold
first_subj = int(subjects[0])
tr_mask = sids != first_subj
va_mask = sids == first_subj
X_tr, y_tr = X[tr_mask], y[tr_mask]
X_val, _ = X[va_mask], y[va_mask]

X_tr_n, _, _, _ = normalise(X_tr, X_val)
loader = DataLoader(
    TensorDataset(torch.from_numpy(X_tr_n), torch.from_numpy(y_tr)),
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

n_cuda = torch.cuda.device_count() if device.type == "cuda" else 0
use_dp = bool(USE_MULTI_GPU and device.type == "cuda" and n_cuda > 1)

model = build_model(arch, n_classes=24).to(device)
model = wrap_data_parallel(model, use_dp, device)
model.train()
criterion = torch.nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=1e-3)

warmup_steps = 2
bench_steps = 10
it = iter(loader)

for _ in range(warmup_steps):
    xb, yb = next(it)
    xb, yb = xb.to(device), yb.to(device)
    optim.zero_grad()
    loss = criterion(model(xb), yb)
    loss.backward()
    optim.step()

if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(bench_steps):
    try:
        xb, yb = next(it)
    except StopIteration:
        it = iter(loader)
        xb, yb = next(it)
    xb, yb = xb.to(device), yb.to(device)
    optim.zero_grad()
    loss = criterion(model(xb), yb)
    loss.backward()
    optim.step()
if device.type == "cuda":
    torch.cuda.synchronize()
dt = time.perf_counter() - t0
sec_per_step = dt / bench_steps

# 3) ETA projection (training only; validation + bookkeeping overhead not included)
est_train_seconds = sec_per_step * steps_per_epoch_all_folds * epochs
overhead_factor = 1.2  # rough cushion for validation/checkpoint/logging
total_seconds = est_train_seconds * overhead_factor

print("--- Pre-training ETA ---")
print(f"Device: {device} | CUDA devices: {n_cuda}")
print(f"Benchmark uses DataParallel (2+ GPU): {use_dp}")
print(f"Data: {DATA_PATH}")
print(f"Samples: {len(X):,} | Subjects (folds): {len(subjects)}")
print(f"Arch: {arch} | Epochs: {epochs} | Batch size: {batch_size}")
print(f"Steps/epoch across all LOSO folds: {steps_per_epoch_all_folds:,}")
print(f"Measured sec/step: {sec_per_step:.4f}")
print(f"Estimated training-only time: {est_train_seconds/60:.1f} min")
print(f"Estimated total time (with overhead): {total_seconds/60:.1f} min (~{total_seconds/3600:.2f} h)")
print("Tip: If ETA is too high, reduce epochs or increase batch size (if memory allows).")

## 5. Run LOSO training

Adjust `--epochs`, `--arch`, `--batch_size` for full runs. Set **`USE_MULTI_GPU = True`** only if the runtime has **2+ GPUs** (uses `train.py --multi_gpu`). Kaggle notebooks normally have a single GPU.

In [ ]:
os.chdir(CODE_DIR)

if SMOKE_TEST:
    epochs = 5
    arch = "resnet"
else:
    epochs = 50
    arch = "resnet"

batch_size = 64
device = "cuda" if torch.cuda.is_available() else "cpu"
import subprocess

extra = ""
if USE_MULTI_GPU and device == "cuda" and torch.cuda.device_count() > 1:
    extra = " --multi_gpu"
    print("Using --multi_gpu across", torch.cuda.device_count(), "device(s)")

cmd = (
    f"python train.py "
    f"--data {DATA_PATH} "
    f"--out_dir {CKPT_DIR} "
    f"--arch {arch} "
    f"--epochs {epochs} "
    f"--batch_size {batch_size} "
    f"--device {device}" + extra
)
print(cmd)
r = subprocess.run(cmd, shell=True, cwd=CODE_DIR)
if r.returncode != 0:
    raise RuntimeError(f"train.py exited with {r.returncode}")

## 6. Zip outputs for download

After the run: **Output** tab → download `sensorloc_outputs.zip`. Copy `best_model_fold*.pt` and `loso_results.txt` into your local `checkpoints/` folder, then run `evaluate.py` locally with the **same** `dataset.npz` you trained on.

In [ ]:
import zipfile

zip_path = os.path.join(WORK_ROOT, "sensorloc_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(CKPT_DIR):
        for f in files:
            fp = os.path.join(root, f)
            z.write(fp, arcname=os.path.relpath(fp, WORK_ROOT))
print("Created:", zip_path)
print("Checkpoint dir contents:", sorted(os.listdir(CKPT_DIR)))

## Local evaluation (after download)

```powershell
cd C:\VS\SensorLoc
python evaluate.py `
  --checkpoint checkpoints/best_model_fold0.pt `
  --data       data/dataset.npz `
  --vote_k     5
```

Use the same `dataset.npz` as on Kaggle. Read `results/eval_summary.json` and `results/confusion_matrix.png` as documented in the project README.